In [3]:
#!pip install git+https://github.com/daviddavo/lightfm

In [4]:
import os
import sys
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

import lightfm
from lightfm import LightFM
from lightfm.data import Dataset
from sklearn.model_selection import train_test_split
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k as lightfm_prec_at_k
from lightfm.evaluation import recall_at_k as lightfm_recall_at_k

print("System version: {}".format(sys.version))
print("LightFM version: {}".format(lightfm.__version__))


System version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
LightFM version: 1.17


La idea ahora será alimentar LightFM con features de los videojuegos, pero no solo categóricas sino que numéricas también. Seguimos la misma estructura de los otros notebooks.

In [5]:
DATA_PATH_RATINGS = "video_game_reviews_with_userid_clean.csv"
DATA_PATH_METADATA = "video_game_reviews.csv"

SEED = 42
TEST_PERCENTAGE = 0.25
K = 10
NO_THREADS = 8
NO_EPOCHS = 25

ratings = pd.read_csv(DATA_PATH_RATINGS)
metadata = pd.read_csv(DATA_PATH_METADATA)

ratings = ratings.rename(columns={
    "user_id": "userID",
    "item_id": "itemID",
    "rating": "rating"
})

ratings["userID"] = ratings["userID"].astype(str)
ratings["itemID"] = ratings["itemID"].astype(str)
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings.drop_duplicates(subset=["userID", "itemID"])
print(f"Ratings: {ratings.shape}")

metadata = metadata.rename(columns={"Game Title": "GameTitle"})
metadata["GameTitle"] = metadata["GameTitle"].astype(str).str.strip()
print(f"Metadata: {metadata.shape}")

Ratings: (39441, 3)
Metadata: (47774, 18)


Ahora creamos las features categóricas igual que el notebook de features, pero ahora hacen un binning numérico, de cuatro cuartiles. Nos pareció que era un enfoque razonable para entrenar el modelo sin llegar a tomar un tiempo excesivo.

Algunas referencias de utilidad
- https://stackoverflow.com/questions/45273731/binning-a-column-with-pandas
- https://pandas.pydata.org/docs/reference/api/pandas.qcut.html

In [6]:
categorical_cols = [
    "Genre", "Platform", "Developer", "Publisher",
    "Game Mode", "Multiplayer", "Requires Special Device",
    "Age Group Targeted"
]

numeric_cols = [
    "Release Year", "Price", "Game Length (Hours)",
    "Graphics Quality", "Soundtrack Quality", "Story Quality", "User Rating"
]

def bin_numeric_feature(s, prefix):
    try:
        s = pd.to_numeric(s, errors="coerce")
        if s.nunique() > 1:
            bins = pd.qcut(s, q=min(4, s.nunique()), duplicates="drop")
            return bins.astype(str).fillna("NA").apply(lambda x: f"{prefix}_bin={x}")
        else:
            return pd.Series([f"{prefix}_bin=NA"] * len(s))
    except Exception:
        return pd.Series([f"{prefix}_bin=NA"] * len(s))


Ya podemos armar las features.

In [7]:
def make_features(row):
    feats = []
    for c in categorical_cols:
        if c in row and pd.notna(row[c]):
            feats.append(f"{c}={str(row[c]).strip()}")
    for c in numeric_cols:
        val = row.get(f"{c}_bin", None)
        if pd.notna(val):
            feats.append(str(val))
    return feats

for c in numeric_cols:
    metadata[f"{c}_bin"] = bin_numeric_feature(metadata[c], c)

metadata["features"] = metadata.apply(make_features, axis=1)

Ahora, volvemos a hacer el mismo procedimiento.

In [8]:
unique_items = sorted(ratings["itemID"].unique())
metadata = metadata.head(len(unique_items)).copy()
metadata["itemID"] = unique_items[:len(metadata)]
metadata["itemID"] = metadata["itemID"].astype(str)

def stratified_user_split(df, test_size=0.25, seed=42):
    train_parts, test_parts = [], []
    for user, grp in df.groupby("userID"):
        if len(grp) < 2:
            train_parts.append(grp)
            continue
        tr, te = train_test_split(grp, test_size=test_size, random_state=seed)
        train_parts.append(tr)
        test_parts.append(te)
    return pd.concat(train_parts), pd.concat(test_parts)

train_df, test_df = stratified_user_split(ratings, test_size=TEST_PERCENTAGE, seed=SEED)
print(f"Train: {train_df.shape}, Test: {test_df.shape}")


Train: (28451, 3), Test: (10990, 3)


Construimos el dataset y entrenamos.

In [9]:
all_users = ratings["userID"].unique()
all_items = ratings["itemID"].unique()
all_features = sorted({f for feats in metadata["features"].dropna() for f in feats})

dataset3 = Dataset()
dataset3.fit(users=all_users, items=all_items, item_features=all_features)

(train_interactions3, _) = dataset3.build_interactions(train_df[["userID", "itemID", "rating"]].values)
(test_interactions3, _)  = dataset3.build_interactions(test_df[["userID", "itemID", "rating"]].values)

def item_features_gen():
    for row in metadata.itertuples(index=False):
        feats = getattr(row, "features")
        if isinstance(feats, list):
            yield (getattr(row, "itemID"), feats)
        else:
            yield (getattr(row, "itemID"), [])

item_features3 = dataset3.build_item_features(item_features_gen())
print(f"Item features shape: {item_features3.shape}")

model3 = LightFM(
    loss='warp',
    no_components=32,
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=np.random.RandomState(SEED)
)

model3.fit(
    train_interactions3,
    item_features=item_features3,
    epochs=NO_EPOCHS,
    num_threads=NO_THREADS
)

Item features shape: (40, 103)


Finalmente, la primera evaluación.

In [10]:
prec3 = lightfm_prec_at_k(
    model3, test_interactions3,
    train_interactions=train_interactions3,
    item_features=item_features3,
    k=K, num_threads=NO_THREADS
).mean()

rec3 = lightfm_recall_at_k(
    model3, test_interactions3,
    train_interactions=train_interactions3,
    item_features=item_features3,
    k=K, num_threads=NO_THREADS
).mean()

print(f"Precision@{K}: {prec3:.4f}")
print(f"Recall@{K}:    {rec3:.4f}")

Precision@10: 0.1222
Recall@10:    0.3273


Luego de investigar sobre cómo mejorar los resultados, notamos que existe el Randomized Search. La idea será implementarlo y probar.
- https://machinelearningmastery.com/hyperparameter-optimization-with-random-search-and-grid-search/

In [11]:
from tqdm import tqdm

# hay que definir el espacio de busqueda
param_distributions = {
    "loss": ["warp", "bpr"],
    "no_components": [16, 32, 48, 64],
    "learning_rate": [0.01, 0.025, 0.05, 0.1],
    "item_alpha": [1e-6, 1e-5, 1e-4],
    "user_alpha": [1e-6, 1e-5, 1e-4],
}

N_ITER = 10     # esta es la cantidad de combinaciones aleatorias
K = 10
EPOCHS = 25
NO_THREADS = 8
SEED = 42
random.seed(SEED)

def evaluate_model(model, train, test, item_feats, k=10):
    prec = lightfm_prec_at_k(model, test, train_interactions=train,
                          item_features=item_feats, k=k,
                          num_threads=NO_THREADS).mean()
    rec = lightfm_recall_at_k(model, test, train_interactions=train,
                      item_features=item_feats, k=k,
                      num_threads=NO_THREADS).mean()
    return float(prec), float(rec)



Ahora comienza la busqueda

In [12]:

results = []
for i in tqdm(range(N_ITER), desc="Randomized Search"):
    params = {k: random.choice(v) for k, v in param_distributions.items()}

    model = LightFM(
        loss=params["loss"],
        no_components=params["no_components"],
        learning_rate=params["learning_rate"],
        item_alpha=params["item_alpha"],
        user_alpha=params["user_alpha"],
        random_state=np.random.RandomState(SEED)
    )

    model.fit(train_interactions3, item_features=item_features3,
              epochs=EPOCHS, num_threads=NO_THREADS)

    prec, rec = evaluate_model(model, train_interactions3,
                               test_interactions3, item_features3, k=K)

    params.update({"precision@10": prec, "recall@10": rec})
    results.append(params)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("precision@10", ascending=False)
display(results_df)

Randomized Search: 100%|██████████| 10/10 [02:37<00:00, 15.73s/it]


,loss,no_components,learning_rate,item_alpha,user_alpha,precision@10,recall@10
4,bpr,16,0.025,0.000100,0.000010,0.126500,0.338317
5,bpr,48,0.025,0.000001,0.000010,0.124400,0.333022
7,bpr,48,0.010,0.000100,0.000010,0.124300,0.333750
1,warp,16,0.010,0.000100,0.000010,0.123833,0.332617
9,bpr,32,0.010,0.000001,0.000100,0.123667,0.331594
2,warp,16,0.010,0.000001,0.000001,0.122833,0.330028
8,warp,64,0.010,0.000100,0.000010,0.121900,0.327406
3,warp,32,0.100,0.000001,0.000010,0.121567,0.326378
0,warp,16,0.050,0.000001,0.000001,0.121133,0.324344
6,warp,16,0.100,0.000001,0.000010,0.119933,0.321872


Con estos resultados, ya se ve que la mejor combinación de las estudiadas es:

In [13]:
model_final = LightFM(
    loss="bpr",
    no_components=16,
    learning_rate=0.025,
    item_alpha=1e-4,
    user_alpha=1e-5,
    random_state=np.random.RandomState(SEED)
)

model_final.fit(
    train_interactions3,
    item_features=item_features3,
    epochs=30,
    num_threads=NO_THREADS
)


Finalmente, la evaluación.

In [14]:
prec_final = lightfm_prec_at_k(
    model_final, test_interactions3,
    train_interactions=train_interactions3,
    item_features=item_features3,
    k=K, num_threads=NO_THREADS
).mean()

rec_final = lightfm_recall_at_k(
    model_final, test_interactions3,
    train_interactions=train_interactions3,
    item_features=item_features3,
    k=K, num_threads=NO_THREADS
).mean()

print(f"Precision@{K}: {prec_final:.4f}")
print(f"Recall@{K}:    {rec_final:.4f}")


Precision@10: 0.1233
Recall@10:    0.3303


In [15]:
user_mapping, _, item_mapping, _ = dataset3.mapping()
print(list(user_mapping.keys())[:5])


['861', '1295', '1131', '1096', '1639']


In [ ]:
# mapeos para convertir entre índices internos y ids originales
user_id_map, user_feat_map, item_id_map, item_feat_map = dataset3.mapping()
inv_item_id_map = {v: k for k, v in item_id_map.items()}
inv_user_id_map = {v: k for k, v in user_id_map.items()}

# recomendar para un usuario (por indice interno de LightFM)
def recommend_for_user(model, user_internal_id, train_interactions, K=10,
                       user_features=None, item_features=None):
    n_users, n_items = train_interactions.shape

    # predice para tds los items
    item_ids = np.arange(n_items)
    scores = model.predict(
        user_ids=user_internal_id,
        item_ids=item_ids,
        user_features=user_features,
        item_features=item_features
    )

    # "enmascara" items ya vistos por ese usuario en el set de training
    known_items = train_interactions.tocsr()[user_internal_id].indices
    scores[known_items] = -np.inf
    valid_mask = np.isfinite(scores)
    num_candidates = int(valid_mask.sum())

    # Top-K, 10 es lo que usamos pero igual es mejor dejarlo parametrizado por si acaso
    K_eff = min(K, num_candidates) if num_candidates > 0 else 0
    top_items = np.argsort(-scores)[:K_eff]

    # convierte indices internos a ids originales
    rec_item_ids = [inv_item_id_map[i] for i in top_items]

    return {
        "requested_K": K,
        "available_candidates": num_candidates,
        "returned_K": K_eff,
        "internal_item_indices": top_items.tolist(),
        "item_ids": rec_item_ids
    }

# como ejemplo, la idea es elegir un usuario válido, por índice interno
example_user_internal_id = 2

res = recommend_for_user(
    model=model_final,
    user_internal_id=example_user_internal_id,
    train_interactions=train_interactions3,
    K=K
)

print(f"usuario interno: {example_user_internal_id} (ID original: {inv_user_id_map.get(example_user_internal_id)})")
print(f"candidatos disponibles (no vistos): {res['available_candidates']}")
print(f"solicitados K={res['requested_K']}, Devueltos: {res['returned_K']}")
print("recomendaciones (itemID):")
for iid in res["item_ids"]:
    print("  -", iid)


usuario interno: 2 (ID original: 1131)
candidatos disponibles (no vistos): 31
solicitados K=10, Devueltos: 10
recomendaciones (itemID):
  - 14
  - 23
  - 16
  - 21
  - 8
  - 35
  - 20
  - 29
  - 12
  - 28


In [17]:
rec_df = pd.DataFrame({"itemID": res["item_ids"]})
rec_df = rec_df.merge(metadata[["itemID", "GameTitle"]], on="itemID", how="left")

print("recomendaciones para el usuario 1131:")
display(rec_df)


recomendaciones para el usuario 1131:


,itemID,GameTitle
0,14,Grand Theft Auto V
1,23,Rocket League
2,16,Just Dance 2024
3,21,Fall Guys
4,8,The Elder Scrolls V: Skyrim
5,35,Tekken 7
6,20,Street Fighter V
7,29,Call of Duty: Modern Warfare 2
8,12,Bioshock Infinite
9,28,Spelunky 2


## Métricas

In [18]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)

def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)

def ndcg_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0

def hit_score_at_k(rec_k, rel_set):
    return 1.0 if any((i in rel_set) for i in rec_k) else 0.0

def map_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0

def diversity_at_k(rec_k_dict, info_videojuegos):
    diversidades = []
    for uid, recs in rec_k_dict.items():
        generos = {info_videojuegos[i][1] for i in recs if i in info_videojuegos}
        diversidades.append(len(generos))
    return np.mean(diversidades) if diversidades else np.nan

Ya definidas, hacemos lo mismo que en los otros notebooks.

In [19]:
itemid2title = dict(metadata[["itemID", "GameTitle"]].drop_duplicates("itemID").values)
itemid2genre = dict(metadata[["itemID", "Genre"]].drop_duplicates("itemID").values)

info_videojuegos = {}
for internal_idx, original_item_id in inv_item_id_map.items():
    title = itemid2title.get(str(original_item_id))
    genre = itemid2genre.get(str(original_item_id))
    info_videojuegos[internal_idx] = (title, genre)

def evaluar_metricas(
    model,
    train_interactions,
    test_interactions,
    K=10,
    num_threads=NO_THREADS,
    info_videojuegos=None
):
    n_users, n_items = train_interactions.shape
    train_csr = train_interactions.tocsr()
    test_csr  = test_interactions.tocsr()

    rec_k_dict = {}
    rows_user = []

    for u in range(n_users):
        # relevantes en test para el usuario
        rel_set = set(test_csr[u].indices)
        item_ids = np.arange(n_items)

        scores = model.predict(
            user_ids=u,
            item_ids=item_ids,
            num_threads=num_threads
        )

        # enmascarar items ya vistos en train
        scores[train_csr[u].indices] = -np.inf

        valid = np.isfinite(scores)
        if not valid.any():
            rec_k = []
        else:
            K_eff = min(K, int(valid.sum()))
            top_idx = np.argsort(-scores)[:K_eff]
            rec_k = list(top_idx)

        rec_k_dict[u] = rec_k

        prec = precision_at_k(rec_k, rel_set)
        rec  = recall_at_k(rec_k, rel_set)
        ndcg = ndcg_at_k(rec_k, rel_set)
        hit  = hit_score_at_k(rec_k, rel_set)
        m_ap = map_at_k(rec_k, rel_set)

        rows_user.append({
            "user_internal": u,
            "relevantes_test": len(rel_set),
            "K_devueltos": len(rec_k),
            "precision@K": prec,
            "recall@K": rec,
            "ndcg@K": ndcg,
            "hit_score@K": hit,
            "map@K": m_ap
        })

    df_users = pd.DataFrame(rows_user)
    global_metrics = {
        "users_evaluated": int(df_users.shape[0]),
        "precision@K": float(df_users["precision@K"].mean()) if not df_users.empty else np.nan,
        "recall@K": float(df_users["recall@K"].mean()) if not df_users.empty else np.nan,
        "ndcg@K": float(df_users["ndcg@K"].mean()) if not df_users.empty else np.nan,
        "hit_score@K": float(df_users["hit_score@K"].mean()) if not df_users.empty else np.nan,
        "map@K": float(df_users["map@K"].mean()) if not df_users.empty else np.nan
    }

    if info_videojuegos is not None and len(info_videojuegos) > 0:
        global_metrics["diversity@K"] = float(diversity_at_k(rec_k_dict, info_videojuegos))
    else:
        global_metrics["diversity@K"] = np.nan

    df_global = pd.DataFrame([global_metrics])

    return df_global

df_global = evaluar_metricas(
    model=model_final,
    train_interactions=train_interactions3,
    test_interactions=test_interactions3,
    K=K,
    num_threads=NO_THREADS,
    info_videojuegos=info_videojuegos
)

display(df_global)

,users_evaluated,precision@K,recall@K,ndcg@K,hit_score@K,map@K,diversity@K
0,3000,0.1244,0.333011,0.227669,0.772333,0.118617,6.422333
